# Karpathy's Micrograd — Deep-ML Checkpoints
Following "Neural Networks: Zero to Hero" (micrograd video).
Each section = one checkpoint: derivatives → chain rule → Value class → neuron → MLP → gradient descent.

***Derivative of a Polynomial***

In [ ]:
def poly_term_derivative(c: float, x: float, n: float) -> float:
    # Your code here
    if n == 0.0:
        return 0.0
    return round(c * n * (x ** (n - 1)),4)

***Partial Derivatives of Multivariable Functions***

In [2]:
import  numpy as np  

def compute_partial_derivatives(func_name, point):
    if len(point) == 2:
        x, y = point
    else:
        x, y, z = point

    if func_name == 'poly2d':
        return (2*x*y + y**2, x**2 + 2*x*y)
    elif func_name == 'exp_sum':
        return (np.exp(x+y), np.exp(x+y))
    elif func_name == 'product_sin':
        return (np.sin(y), x*np.cos(y))
    elif func_name == 'poly3d':
        return (2*x*y, x**2 + z**2, 2*y*z)
    elif func_name == 'squared_error':
        return (2*(x-y), -2*(x-y))
    


***Chain Rule for Composite Functions***

In [ ]:
import numpy as np

def compute_chain_rule_gradient(functions: list[str], x: float) -> float:
	"""
	Compute derivative of composite functions using chain rule.
	
	Args:
		functions: List of function names (applied right to left)
		          Available: 'square', 'sin', 'exp', 'log'
		x: Point at which to evaluate derivative
	
	Returns:
		Derivative value at x
	
	Example:
		['sin', 'square'] represents sin(xÂ²)
		['exp', 'sin', 'square'] represents exp(sin(xÂ²))
	"""
	# Your code here
	# Forward pass 
	derivs = {
    'square': lambda x: 2*x,
    'sin': np.cos,
    'exp': np.exp,
    'log': lambda x: 1/x,
}
	funcs = {
    'square': lambda x: x**2,
    'sin': np.sin,
    'exp': np.exp,
    'log': np.log,
}
	u = x
	intermediates = [x]
	for f in reversed(functions):
		u = funcs[f](u)
		intermediates.append(u)
    
    # back pass 
	result = 1.0
	for i , f in enumerate(reversed(functions)):
		result *= derivs[f](intermediates[i])
	return round(result , 6)
		
    
		
       

	

***Gradient Direction and Magnitude***


In [ ]:
import numpy as np

def gradient_direction_magnitude(gradient: list) -> dict:
	"""
	Calculate the magnitude and direction of a gradient vector.
	
	Args:
		gradient: A list representing the gradient vector
	
	Returns:
		Dictionary containing:
		- magnitude: The L2 norm of the gradient
		- direction: Unit vector in direction of steepest ascent
		- descent_direction: Unit vector in direction of steepest descent
	"""
	# Your code here
	gradient = np.array(gradient)
	Magnitude = np.linalg.norm(gradient)
	if Magnitude == 0:
    	return {
        'magnitude': 0.0,
        'direction': [0.0] * len(gradient),
        'descent_direction': [0.0] * len(gradient)
    }
	Direction = gradient / Magnitude
	Descent_direction = -Direction
	return {'magnitude': Magnitude, 'direction': Direction.tolist(), 'descent_direction': Descent_direction.tolist()}



***Implement the Tanh Activation Function***


In [ ]:
import math

def tanh(x: float) -> float:
	"""
	Implements the Tanh (hyperbolic tangent) activation function.

	Args:
		x (float): Input value

	Returns:
		float: The tanh of the input, rounded to 4 decimal places
	"""
	# Your code here
	return (math.exp(x) - math.exp(-x)) / (math.exp(x) + math.exp(-x))

***Derivatives of Activation Functions***

In [ ]:
import numpy as np 
import math 

def activation_derivatives(x: float) -> dict[str, float]:
	"""
	Compute the derivatives of Sigmoid, Tanh, and ReLU at a given point x.
	
	Args:
		x: Input value
		
	Returns:
		Dictionary with keys 'sigmoid', 'tanh', 'relu' and their derivative values
	"""
	# Your code here
	# {'sigmoid': 0.25, 'tanh': 1.0, 'relu': 0.0} 
	def sigmoid_dev(x):
		# 
		sigmoid = (1 / (1 + np.exp(-x)))
		return float(sigmoid * (1 - sigmoid))
	
	def tanh_dev(x):
		tanh = ((math.exp(x) - math.exp(-x)) / (math.exp(x) + math.exp(-x)))
		return float(1 - tanh**2)
	
	def relu_dev(x):
		if x > 0 :
			return 1.0
		else:
			return 0.0
	
	return {'sigmoid': sigmoid_dev(x), 'tanh': tanh_dev(x), 'relu': relu_dev(x)}



***Implementing Basic Autograd Operations***

In [ ]:
class Value:
	def __init__(self, data, _children=(), _op=''):
		self.data = data
		self.grad = 0
		self._backward = lambda: None
		self._prev = set(_children)
		self._op = _op
	def __repr__(self):
		def fmt(x):
			return int(x) if float(x).is_integer() else round(x, 4)
		return f"Value(data={fmt(self.data)}, grad={fmt(self.grad)})"

	def __add__(self, other):
		 # Implement addition here
		out = Value(self.data +  other.data,(self,other), '+')

		def _backward():
			self.grad += 1.0 * out.grad
			other.grad += 1.0 * out.grad
		out._backward = _backward
		return out 
	

	def __mul__(self, other):
		# Implement multiplication here
		out = Value(self.data *  other.data,(self,other), '*')

		def _backward():
			self.grad += other.data * out.grad
			other.grad += self.data * out.grad
		out._backward = _backward
		return out 

	def relu(self):
		out = Value(max(0, self.data), (self,), 'relu')
		def _backward():
			self.grad += (out.data > 0) * out.grad
		out._backward = _backward
		return out

	def backward(self):
		topo = []
		visited = set()
		def build_topo(v):
			if v not in visited:
				visited.add(v)
				for child in v._prev:
					build_topo(child)
				topo.append(v)
		build_topo(self)
		self.grad = 1.0
		for node in reversed(topo):
			node._backward()

***Implement the Softplus Activation Function***

In [ ]:
import numpy as np 


def softplus(x: float) -> float:
	"""
	Compute the softplus activation function.

	Args:
		x: Input value

	Returns:
		The softplus value: log(1 + e^x)
	"""
	# Your code here
	val = np.log(1 + np.exp(x))
	return round(val,4)

***Lab :PyTorch: Build a Complete Training Loop***


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

def train_model(model, X_train, y_train, X_val, y_val, epochs, batch_size, lr):
    history = []

    # setup
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    n_samples = X_train.shape[0]

    for epoch in range(1, epochs + 1):          # 1-indexed, as the spec wants

        # ---- train ----
        model.train()

        # shuffle (X and y with the SAME perm — flashcards stay married)
        perm = torch.randperm(n_samples)
        X_shuffled = X_train[perm]
        y_shuffled = y_train[perm]

        epoch_loss = 0.0
        n_batches = 0

        # batch loop
        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i : i + batch_size]
            y_batch = y_shuffled[i : i + batch_size]

            # the training step — yours, unchanged
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()           # .item() → plain float, no memory leak
            n_batches += 1

        # ---- validate ----
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = criterion(val_logits, y_val).item()
            preds = val_logits.argmax(dim=1)
            val_accuracy = (preds == y_val).float().mean().item()

        # ---- record ----
        history.append({
            'epoch': epoch,
            'train_loss': epoch_loss / n_batches,
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
        })

    return history

***Single Neuron***


In [ ]:
import math
import numpy as np 


def single_neuron_model(features: list[list[float]], labels: list[int], weights: list[float], bias: float) :
	# Your code here
	z=np.dot(features,weights)+bias
	sigma = 1 / (1 + np.exp(-z))
	mse = np.mean((sigma - labels)**2)
	
	return np.round(sigma,4).tolist(), np.round(mse,4).tolist()



***Calculate Number of Parameters in Neural Network***

In [ ]:
def calculate_parameters(layers: list[dict]) -> int:
    total = 0
    for layer in layers:

        if layer['type'] == 'dense':
            w = layer['input_size'] * layer['output_size']
            if layer.get('bias', True):        # missing key → defaults to True
                w += layer['output_size']
            total += w

        elif layer['type'] == 'conv2d':
            w = layer['in_channels'] * layer['out_channels'] * layer['kernel_size']**2
            if layer.get('bias', True):
                w += layer['out_channels']
            total += w

    return total

***Linear Regression Using Gradient Descent***

In [ ]:
import numpy as np

def linear_regression_gradient_descent(X: np.ndarray, y: np.ndarray, alpha: float, iterations: int) -> np.ndarray:
    """
    Perform linear regression using gradient descent.

    Args:
        X: Feature matrix of shape (m, n) where first column is all ones (for intercept)
        y: Target vector of shape (m,)
        alpha: Learning rate
        iterations: Number of gradient descent iterations
    
    Returns:
        Learned weights as a 1D array of shape (n,)
    """
    m, n = X.shape
    y = y.reshape(-1, 1)  # Ensure y is a column vector
    theta = np.zeros((n, 1))  # Initialize weights to zeros
    
    for interation in range(iterations):
        gradient = (1/m) * X.T @ (X @ theta -y)
        theta = theta - alpha * gradient
    # Your code here: implement gradient descent
    
    return theta.flatten()

***Implement Weight Decay as L2 Regularization***

In [ ]:
def apply_weight_decay(parameters: list[list[float]], gradients: list[list[float]], 
                       lr: float, weight_decay: float, apply_to_all: list[bool]) -> list[list[float]]:
	"""
	Apply weight decay (L2 regularization) to parameters.
	
	Args:
		parameters: List of parameter arrays
		gradients: List of gradient arrays
		lr: Learning rate
		weight_decay: Weight decay factor
		apply_to_all: Boolean list indicating which parameter groups get weight decay
	
	Returns:
		Updated parameters
	"""
	# Your code here
	
	for i in range(len(parameters)):
		if apply_to_all[i]: 
			for j in range(len(gradients[i])):
				# wnew​=w−η⋅g−η⋅λ⋅w 
				w_new = parameters[i][j] - (lr * gradients[i][j]) - (lr * weight_decay * parameters[i][j] )
				parameters[i][j] = w_new
		else :
			for j in range(len(gradients[i])): 
				# w - lr * g 
				w_new = parameters[i][j] - (lr * gradients[i][j])
				parameters[i][j] = w_new
	return parameters

***Single Neuron with Backpropagation***

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_neuron(features, labels, initial_weights, initial_bias, learning_rate, epochs):
    features = np.array(features)
    labels = np.array(labels)
    w = np.array(initial_weights, dtype=float).copy()
    b = initial_bias
    mse_values = []

    for epoch in range(epochs):
        z = np.dot(features, w) + b
        y_hat = sigmoid(z)
        MSE = np.mean((y_hat - labels)**2)
        delta = (y_hat - labels) * y_hat * (1 - y_hat)
        dL_dw = 2 * np.dot(features.T, delta) / len(features)
        dL_db = 2 * np.mean(delta)

        w = w - learning_rate * dL_dw
        b = b - learning_rate * dL_db

        mse_values.append(round(MSE, 4))

    return np.round(w, 4), round(b, 4), mse_values